In [1]:
import time
import ast
from itertools import chain
from collections import Counter

import pandas as pd

In [2]:
from pathlib import Path

DATASET_CSV_PATH = Path(Path('').absolute(), '..',
                        'datasets', 'news', 'filtered_all.csv')

In [3]:
filtered_data = pd.read_csv(DATASET_CSV_PATH)

In [4]:
display(filtered_data.head())

,content,heading,label
0,नेपालका लागि टाटा मोटर्सको एकमात्र आधिकारिक बि...,टाटा मोटर्सले ‘डिसेम्बर ब्लेजिङ्ग एक्सचेञ्ज’ क...,auto
1,टाटा मोटर्स नेपालको आधिकारिक विक्रेता सिप्रदी ...,"टाटा मोटर्सको ‘टाटा एचफाइभ वन डे वन्डर’, कसरी ...",auto
2,"२३ मंसिर, काठमाडौं । जगदम्बा मोटर्सले मलेसियाक...",जगदम्बा मोटर्सले प्रोटोन नेपाल भित्र्याउँदै,auto
3,"२३ मंसिर, काठमाडौं । नेपालको लागि बीएमडब्ल्यू ...",नेपालमा बीएमडब्ल्यूको बिक्री सुरु,auto
4,"१९ मंसिर, काठमाडौं । लक्ष्मी ग्रुप नेपालको अन्...",लक्ष्मी प्रिमियम मोटर्सलाई नेपालको लागि बीएमडब...,auto


In [9]:
from snowballstemmer import NepaliStemmer

stemmer = NepaliStemmer()

def tokenizer(text):
    tokenized_word = []
    stem_list = stemmer.stemWords(text.split())
    tokenized_words = list(filter(is_relevant_nepali_word,
                                  stem_list))

    return tokenized_word


def is_relevant_nepali_word(stem_word):
    if len(stem_word) > 3:
        for letter in stem_word:
            if not 0x900 <= ord(letter) <= 0x97F:
                return False
            
        return True
    else:
        return False

In [14]:
filtered_data['text'] = (filtered_data['heading'] + ' ' + filtered_data['content'])

In [16]:
from snowballstemmer import NepaliStemmer

nep_stem = NepaliStemmer()

filtered_data['text'] = filtered_data['text'].apply(
    lambda t: nep_stem.stemWords(t.split()))

In [17]:
filtered_data['text']

0        [टाटा, मोटर्स, ‘डिसेम्बर, ब्लेजिङ्ग, एक्सचेञ्ज...
1        [टाटा, मोटर्स, ‘टाटा, एचफाइभ, वन, डे, वन्डर’,,...
2        [जगदम्बा, मोटर्स, प्रोटोन, नेपाल, भित्र्याउँ, ...
3        [नेपाल, बीएमडब्ल्यू, बिक्री, सुरु, २३, मंसिर,,...
4        [लक्ष्मी, प्रिमियम, मोटर्स, नेपाल, लागि, बीएमड...
                               ...                        
96539    [पर्यटन, क्षेत्र, नखो, चरणवद्ध, आन्दोलन, पर्यट...
96540    [लकडाउन, खुले, कुश्मा, लठ्ठे, पिङ, सन्चलनमा,, ...
96541    [पोखरा, पर्यटकीयस्थल, सूचना, विभिन्न, भाषा, का...
96542    [गण्डकी, खुल्, पर्यटन,, तर, खुलेनन्, व्यवसायी,...
96543    [गोरु, जुधाइ, एक, बहाना, :, प्रेम, प्रस्ताव, म...
Name: text, Length: 96544, dtype: object

In [18]:
final_data = pd.DataFrame(filtered_data,
                          columns=['text','label'])

final_data.iloc[0, :]

text     [टाटा, मोटर्स, ‘डिसेम्बर, ब्लेजिङ्ग, एक्सचेञ्ज...
label                                                 auto
Name: 0, dtype: object

In [19]:
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np


def stratified_train_test_split(texts, labels, test_size, random_state):
    sss = StratifiedShuffleSplit(n_splits=1,
                                 test_size=test_size,
                                 random_state=random_state)
    
    for train_idx, test_idx in sss.split(texts, labels):
        train_texts = texts.iloc[train_idx]
        train_labels = labels.iloc[train_idx]
        test_texts = texts.iloc[test_idx]
        test_labels = labels.iloc[test_idx]

    return ((train_texts.tolist(), train_labels.to_numpy()),
            (test_texts.tolist(), test_labels.to_numpy()))

In [20]:
(train_texts, train_labels), (test_texts, test_labels) = stratified_train_test_split(
    final_data['text'], final_data['label'],
    test_size=0.2, random_state=42)

In [21]:
train_texts[:5]

[['मोटरसाइकल',
  'ठक्करबाट',
  '८२',
  'वर्षीय',
  'बृद्धा',
  'मृत्यु',
  'मोरङ',
  'बेलबारी',
  'नगरपालिका–१०',
  'स्थित',
  'पूर्व–पश्चिम',
  'राजमार्ग',
  'मोटरसाइकल',
  'ठक्करबाट',
  'एक',
  'वृद्ध',
  'मृत्यु',
  'भ',
  'छ।',
  'पूर्वबाट',
  'पश्चिम',
  'जाँ',
  'गर',
  'स',
  '४',
  'प',
  '८२२१',
  'नं',
  'मोटरसाइकल',
  'पैदलयात्री',
  'सोही',
  'ठाउँ',
  '८२',
  'वर्षीय',
  'लक्ष्मण',
  'थापामगर',
  'बिहीबार',
  'दिउँसो',
  'ठक्कर',
  'द',
  'थियो।',
  'मोटरसाइकल',
  'ठक्करबाट',
  'गम्भीर',
  'घाइते',
  'मगर',
  'न्युरो',
  'अस्पताल,',
  'विराटनगर',
  'उपचार',
  'क्रम',
  'बिहीबार',
  'साँझ',
  'मृत्यु',
  'भ',
  'इला',
  'प्रहरी',
  'कार्यालय',
  'बेलवारी',
  'प्रहरी',
  'निरीक्षक',
  'रामजी',
  'कटुवाल',
  'जानकारी',
  'दिए।',
  'ठक्कर',
  'दि',
  'मोटरसाइकल',
  'र',
  'त्यस',
  'चालक',
  'उदयपुर',
  'त्रियुगा',
  'नगरपालिका–',
  '११',
  'पीपलचोक',
  'बस्',
  '२८',
  'वर्षीय',
  'विनोद',
  'दाहाल',
  'प्रहरी',
  'नियन्त्रण',
  'ल',
  'छ।',
  'रासस'],
 ['ताप्लेजुङ',
  'पर्यट

In [22]:
len(train_texts), train_labels.shape, len(test_texts), test_labels.shape

(77235, (77235,), 19309, (19309,))

In [35]:
final_data['label'].value_counts() / len(final_data)

entertainment          0.092517
economy                0.085298
crime                  0.073562
international          0.066488
sports                 0.061588
health                 0.054669
bank                   0.045109
literature             0.041857
corporate              0.034565
science_technology     0.031022
opinion                0.031001
migration              0.030846
politics               0.030515
accidents              0.029696
hollywood-bollywood    0.029282
national               0.029127
tourism                0.028609
nepal-congress         0.014802
film                   0.014263
prachanda              0.013610
auto                   0.013155
music                  0.012067
interview              0.011435
employment             0.009043
cricket                0.008784
gassip                 0.008639
interesting            0.007675
religion               0.007106
science-technology     0.006971
model                  0.006153
lifestyle              0.006122
educatio

In [14]:
test_labels.value_counts() / len(test_labels)

entertainment          0.092548
economy                0.085297
crime                  0.073593
international          0.066497
sports                 0.061578
health                 0.054690
bank                   0.045108
literature             0.041846
corporate              0.034543
science_technology     0.031022
opinion                0.031022
migration              0.030866
politics               0.030504
accidents              0.029675
hollywood-bollywood    0.029261
national               0.029106
tourism                0.028588
nepal-congress         0.014812
film                   0.014242
prachanda              0.013621
auto                   0.013154
music                  0.012067
interview              0.011445
employment             0.009063
cricket                0.008804
gassip                 0.008649
interesting            0.007665
religion               0.007095
science-technology     0.006992
model                  0.006163
education              0.006111
lifestyl

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectKBest
from sklearn.preprocessing import OneHotEncoder
import numpy as np


def ngram_vectorizer(train_texts, train_labels, test_texts, *,
                     ngram_range, tokenizer, analyzer, min_df,
                     max_features, top_k, stop_words=None):
    vectorizer = TfidfVectorizer(ngram_range=ngram_range,
                                 lowercase=False,
                                 tokenizer=tokenizer,
                                 analyzer=analyzer,
                                 min_df=min_df,
                                 max_features=max_features,
                                 dtype=np.float32,
                                 stop_words=stop_words)

    x_train = vectorizer.fit_transform(train_texts, train_labels)
    x_test = vectorizer.transform(test_texts)
    
#     selector = SelectKBest(f_classif, k=top_k)
    
#     x_train = selector.fit_transform(x_train, train_labels)
#     x_test = selector.transform(x_test)
    
    return x_train, x_test, vectorizer


def to_one_hot(train_labels, test_labels):
    oh_enc = OneHotEncoder(dtype=np.float32)
    
    y_train = oh_enc.fit_transform(train_labels.reshape(-1, 1))
    y_test = oh_enc.transform(test_labels.reshape(-1, 1))
    
    return y_train, y_test, oh_enc.categories_

In [19]:
x_train, x_test, tfidf_vectorizer = ngram_vectorizer(train_texts,
                                                     train_labels,
                                                     test_texts,
                                                     ngram_range=(1, 1),
                                                     tokenizer=lambda l: l[0].split(),
                                                     analyzer='word',
                                                     min_df=2,
                                                     max_features=20000,
                                                     top_k=200,
                                                     stop_words=None)

y_train, y_test, _ = to_one_hot(train_labels, test_labels)

In [ ]:
words = tfidf_vectorizer.get_feature_names_out()
top_10_indices = x_train.toarray().argpartition(kth=-10)[-10:]
words[top_10_indices[0]], train_texts[0]

In [20]:
np.argpartition?

In [23]:
from rake_nltk import Rake
import nltk

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/workerap/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/workerap/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [38]:
from nltk.corpus import stopwords

nepali_stopwords = stopwords.words('nepali')
nepali_puncts = set([',', '‘', '’', '।', '(', ')', '–'])

In [39]:
nepali_stopwords

['छ',
 'र',
 'पनि',
 'छन्',
 'लागि',
 'भएको',
 'गरेको',
 'भने',
 'गर्न',
 'गर्ने',
 'हो',
 'तथा',
 'यो',
 'रहेको',
 'उनले',
 'थियो',
 'हुने',
 'गरेका',
 'थिए',
 'गर्दै',
 'तर',
 'नै',
 'को',
 'मा',
 'हुन्',
 'भन्ने',
 'हुन',
 'गरी',
 'त',
 'हुन्छ',
 'अब',
 'के',
 'रहेका',
 'गरेर',
 'छैन',
 'दिए',
 'भए',
 'यस',
 'ले',
 'गर्नु',
 'औं',
 'सो',
 'त्यो',
 'कि',
 'जुन',
 'यी',
 'का',
 'गरि',
 'ती',
 'न',
 'छु',
 'छौं',
 'लाई',
 'नि',
 'उप',
 'अक्सर',
 'आदि',
 'कसरी',
 'क्रमशः',
 'चाले',
 'अगाडी',
 'अझै',
 'अनुसार',
 'अन्तर्गत',
 'अन्य',
 'अन्यत्र',
 'अन्यथा',
 'अरु',
 'अरुलाई',
 'अर्को',
 'अर्थात',
 'अर्थात्',
 'अलग',
 'आए',
 'आजको',
 'ओठ',
 'आत्म',
 'आफू',
 'आफूलाई',
 'आफ्नै',
 'आफ्नो',
 'आयो',
 'उदाहरण',
 'उनको',
 'उहालाई',
 'एउटै',
 'एक',
 'एकदम',
 'कतै',
 'कम से कम',
 'कसै',
 'कसैले',
 'कहाँबाट',
 'कहिलेकाहीं',
 'का',
 'किन',
 'किनभने',
 'कुनै',
 'कुरा',
 'कृपया',
 'केही',
 'कोही',
 'गए',
 'गरौं',
 'गर्छ',
 'गर्छु',
 'गर्नुपर्छ',
 'गयौ',
 'गैर',
 'चार',
 'चाहनुहुन्छ',
 'चाहन्छु',
 'चाहिए

In [40]:
from rake_nltk import Rake

rake = Rake(language='nepali',
            punctuations=nepali_puncts,
            include_repeated_phrases=False,
            word_tokenizer=lambda s: s.split(' '),
            sentence_tokenizer=lambda s: s.split('।'))

text = ' '.join(train_texts[0])
rake.extract_keywords_from_text(text)

In [38]:
x_train.todense()[0]

matrix([[0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.04500891,
         0.        , 0.        , 0.        , 0.        , 0.        ,
         0.06822883, 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.06728576, 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.06339376, 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.07170969, 0.3447513 , 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.02073   ,
         0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0

In [ ]:
class tfidf():
    def __init__(self, df,num_keywords):
        self.df = df
#         print("len of df: ", len(df))
        self.corpus = self.df["all_heading_content"].tolist()
        self.documents = self.df["all_heading_content_list"].tolist()
        self.n_documents = len(self.documents)
        self.flatten_list = list(chain.from_iterable(self.documents))
        words = Counter(self.flatten_list)
        # print('words before dic: ',words)
        words = {k:v for k,v in words.items()}
        self.word2idx = {o:i for i,o in enumerate(words)}
        self.idx2word = {i:o for i,o in enumerate(words)}
        print('Total word count: ',len(words))
#         self.countvectorizer = CountVectorizer(analyzer = 'word',min_df = 5,max_features = 100000,tokenizer = self.fake_tokenizer)
        self.tfidfvectorizer = TfidfVectorizer(analyzer = 'word',min_df =3,max_features = 100000,tokenizer = self.fake_tokenizer)
        self.top_n = int(num_keywords)
        
        
    def fake_tokenizer(self,text):
        return text.split()
    
    
    def get_keywords(self):
#         count_wm =  self.countvectorizer.fit_transform(self.corpus)
        tfidf_wm =  self.tfidfvectorizer.fit_transform(self.corpus)

#         print('count_wm.shape,tfidf_wm.shape :',count_wm.shape,tfidf_wm.shape)
        
#         count_tokens = self.countvectorizer.get_feature_names()
        tfidf_tokens = self.tfidfvectorizer.get_feature_names()
        
#         df_countvect = pd.DataFrame(data = count_wm.toarray(),columns = count_tokens)
        df_tfidfvect = pd.DataFrame(data = tfidf_wm.toarray(),columns = tfidf_tokens)
        
        keywords_df = pd.DataFrame({n: df_tfidfvect[:].T[col].nlargest(self.top_n).index.tolist() for n, col in enumerate(df_tfidfvect[:].T)}).T
        keywords_df['keywords'] = keywords_df.values.tolist()
#         print(keywords_df['keywords'])
        return keywords_df['keywords']

In [ ]:
for grouped_df in all_grouped_df:
    if len(grouped_df) > 100:
        print("len of df before: ", len(grouped_df))
        print(f"Finding keywords for {grouped_df.label.unique()[0]}")
        grouped_df.reset_index(inplace=True)
        tfidf_obj = tfidf(grouped_df,20)
        keyword_df = tfidf_obj.get_keywords()
        
        grouped_df['keywords'] = keyword_df
        grouped_df.dropna(inplace=True)
        print("len of df after: ", len(grouped_df))
        print("----------------------------------------------------------------------")
#         print(grouped_df.head(5))

In [ ]:
import pickle

In [ ]:
with open('processed_data/df_with_tfidf_keywords.pickle', 'wb') as f:
    pickle.dump(combined_df, f)